In [ ]:
"""
Minimal multi-horizon classification labels for deep day-by-day training.
Only the fixed-threshold path used by the stable early-stop pipeline is kept.
"""

import numpy as np
import pandas as pd
from typing import List

DEFAULT_HORIZONS: List[int] = [10, 20, 50, 100]


def _mid_price_array(df: pd.DataFrame, dtype=np.float64) -> np.ndarray:
    bp = df["bid_price_1"].to_numpy(dtype=dtype, copy=False)
    ap = df["ask_price_1"].to_numpy(dtype=dtype, copy=False)
    return (ap + bp) * 0.5


def _future_shift(arr: np.ndarray, k: int) -> np.ndarray:
    n = len(arr)
    out = np.full(n, np.nan, dtype=arr.dtype)
    if k < n:
        out[: n - k] = arr[k:]
    return out


def _smoothed_mid(m: np.ndarray, k: int) -> np.ndarray:
    n = len(m)
    out = np.full(n, np.nan, dtype=m.dtype)
    if k >= n:
        return out

    cumsum = np.cumsum(m, dtype=m.dtype)
    out[: n - k] = (cumsum[k:] - cumsum[:-k]) / k
    return out


def make_fixed_threshold_classification_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    alpha: float = 0.002,
    use_smoothing: bool = True,
    dtype=np.float64,
) -> pd.DataFrame:
    if horizons is None:
        horizons = DEFAULT_HORIZONS

    m = _mid_price_array(df, dtype)
    n = len(m)
    out = np.empty((n, len(horizons)), dtype=dtype)

    for i, k in enumerate(horizons):
        future_m = _smoothed_mid(m, k) if use_smoothing else _future_shift(m, k)
        pct_change = (future_m - m) / m

        labels = np.full(n, np.nan, dtype=dtype)
        up_mask = pct_change > alpha
        down_mask = pct_change < -alpha
        mid_mask = (~np.isnan(pct_change)) & (~up_mask) & (~down_mask)

        labels[up_mask] = 2
        labels[down_mask] = 0
        labels[mid_mask] = 1
        out[:, i] = labels

    cols = [f"label_{k}" for k in horizons]
    return pd.DataFrame(out, index=df.index, columns=cols)


In [ ]:
# Colab/local setup for deep multi-epoch day-by-day training
import os
from pathlib import Path


def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


IN_COLAB = is_colab()

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/multi-horizon-ofi')
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = str(PROJECT_ROOT / 'data' / 'processed')
WEIGHTS_DIR = str(PROJECT_ROOT / 'model_weights')
RESULTS_DIR = str(PROJECT_ROOT / 'results')

Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"DATA_DIR not found: {DATA_DIR}. "
        "In Colab, ensure Drive is mounted and the repo is at /content/drive/MyDrive/multi-horizon-ofi"
    )

tickers = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
    and any(f.endswith('.parquet') for f in os.listdir(os.path.join(DATA_DIR, d)))
])

print(f"IN_COLAB    : {IN_COLAB}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"WEIGHTS_DIR : {WEIGHTS_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"Tickers     : {tickers}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: Deep Learning Streaming Setup (Memory-safe)
# ══════════════════════════════════════════════════════════════════════════════
import os
import gc
import json
import time
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEEP_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEEP_DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


DEEP_RAW_LOB_10_COLS = globals().get(
    'RAW_LOB_10_COLS',
    [
        f"{side}_{field}_{lvl}"
        for lvl in range(1, 11)
        for side, field in (
            ('ask', 'price'),
            ('ask', 'size'),
            ('bid', 'price'),
            ('bid', 'size'),
        )
    ],
)

DEEP_HORIZONS = list(globals().get('DEFAULT_HORIZONS', [10, 20, 50, 100]))

DEEP_CONFIG = {
    'data_dir': str(globals().get('DATA_DIR', os.path.join(os.getcwd(), 'data', 'processed'))),
    'weights_dir': str(globals().get('WEIGHTS_DIR', os.path.join(os.getcwd(), 'model_weights'))),
    'results_dir': str(globals().get('RESULTS_DIR', os.path.join(os.getcwd(), 'results'))),
    'ticker': None,
    'tickers': None,
    'horizons': DEEP_HORIZONS,
    'seq_len': 100,
    'alpha': float(globals().get('CONFIG', {}).get('alpha', 0.00005) if 'CONFIG' in globals() else 0.00005),
    'train_file_fraction': 0.8,
    'max_files_per_ticker': 0,
    'base_subsample': 16,
    'high_pressure_subsample': 16,
    'critical_pressure_subsample': 32,
    'max_rows_per_day_train': 8000,
    'max_rows_per_day_eval': 10000,
    'batch_size': 32,
    'epochs': 1,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    'num_workers': 0,
    'amp': True,
    'seed': 42,
    'run_architectures': [
        'dilated_transformer',
        'hybrid_cnn_inception_lstm',
        'seq2seq_attn',
    ],
}


@dataclass
class DayStats:
    ticker: str
    file_name: str
    rows_raw: int
    rows_kept: int
    step: int
    max_rows: int


def deep_set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# -----------------------------
# File discovery/splitting
# -----------------------------
def _deep_discover_tickers(data_dir: str) -> List[str]:
    tickers: List[str] = []
    if not os.path.isdir(data_dir):
        return tickers

    for d in sorted(os.listdir(data_dir)):
        p = os.path.join(data_dir, d)
        if not os.path.isdir(p):
            continue
        has_parquet = any(f.endswith('.parquet') for f in os.listdir(p))
        if has_parquet:
            tickers.append(d)
    return tickers


def _deep_resolve_tickers(cfg: dict) -> List[str]:
    if cfg.get('tickers'):
        return list(cfg['tickers'])
    if cfg.get('ticker'):
        return [cfg['ticker']]
    return _deep_discover_tickers(cfg['data_dir'])


def _deep_collect_files_by_ticker(data_dir: str, tickers: List[str], max_files_per_ticker: int) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for ticker in tickers:
        t_dir = os.path.join(data_dir, ticker)
        if not os.path.isdir(t_dir):
            continue

        files = sorted(
            os.path.join(t_dir, f)
            for f in os.listdir(t_dir)
            if f.endswith('.parquet')
        )

        if max_files_per_ticker > 0:
            files = files[:max_files_per_ticker]

        if files:
            out[ticker] = files
    return out


def _deep_split_train_eval_files(files_by_ticker: Dict[str, List[str]], train_frac: float) -> Tuple[List[Tuple[str, str]], List[Tuple[str, str]]]:
    train_files: List[Tuple[str, str]] = []
    eval_files: List[Tuple[str, str]] = []

    for ticker, files in files_by_ticker.items():
        n = len(files)
        if n <= 1:
            n_train = 1
        else:
            n_train = int(np.floor(n * train_frac))
            n_train = max(1, min(n - 1, n_train))

        for i, p in enumerate(files):
            if i < n_train:
                train_files.append((ticker, p))
            else:
                eval_files.append((ticker, p))

    return train_files, eval_files


# -----------------------------
# Day-level subsampling rules
# -----------------------------
def _deep_choose_subsample(cfg: dict) -> int:
    return int(max(1, cfg.get('base_subsample', 16)))


def _deep_choose_max_rows(cfg: dict, is_train: bool) -> int:
    key = 'max_rows_per_day_train' if is_train else 'max_rows_per_day_eval'
    default = 8000 if is_train else 10000
    return int(max(1, cfg.get(key, default)))


# -----------------------------
# Day dataset builder
# -----------------------------
def _deep_make_loader(dataset: Dataset, cfg: dict, is_train: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=int(cfg['batch_size']),
        shuffle=bool(is_train),
        num_workers=int(cfg['num_workers']),
        pin_memory=(DEEP_DEVICE.type == 'cuda'),
        drop_last=False,
    )


def _deep_update_confusion(cm: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    idx = y_true.astype(np.int64, copy=False) * 3 + y_pred.astype(np.int64, copy=False)
    cm += np.bincount(idx, minlength=9).reshape(3, 3)


def _deep_metrics_from_confusion(cm: np.ndarray, h: int) -> Dict[str, float]:
    cm = cm.astype(np.float64, copy=False)
    tp = np.diag(cm)
    support = cm.sum(axis=1)
    pred_count = cm.sum(axis=0)
    total = support.sum()

    if total == 0:
        return {
            f'h{h}_accuracy': 0.0,
            f'h{h}_f1_macro': 0.0,
            f'h{h}_f1_weighted': 0.0,
            f'h{h}_precision_macro': 0.0,
            f'h{h}_recall_macro': 0.0,
        }

    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(
        2.0 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    return {
        f'h{h}_accuracy': float(tp.sum() / total),
        f'h{h}_f1_macro': float(np.mean(f1)),
        f'h{h}_f1_weighted': float(np.sum(f1 * support) / total),
        f'h{h}_precision_macro': float(np.mean(precision)),
        f'h{h}_recall_macro': float(np.mean(recall)),
    }


def _deep_cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"Deep training device: {DEEP_DEVICE}")
print(f"Deep data dir: {DEEP_CONFIG['data_dir']}")
print(f"Horizons: {DEEP_CONFIG['horizons']}")
print('Cell 7 ready.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Deep Model Definitions
# 1) Dilated CNN + Masked Multihead Attention (Transformer-style)
# 2) Hybrid CNN + Inception + LSTM
# 3) Seq2Seq + Attention (DeepLOB encoder + LSTM decoder)
# ══════════════════════════════════════════════════════════════════════════════

class CausalDilatedConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int, dropout: float = 0.1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, dilation=dilation, padding=self.pad)
        self.bn = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.res = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.conv(x)
        if self.pad > 0:
            y = y[..., :-self.pad]
        y = self.drop(F.gelu(self.bn(y)))
        r = self.res(x)
        if r.shape[-1] != y.shape[-1]:
            r = r[..., -y.shape[-1]:]
        return y + r


class DilatedMaskedTransformer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        horizon_count: int,
        num_classes: int = 3,
        d_model: int = 96,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.15,
        max_len: int = 1024,
    ):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes = num_classes

        self.input_proj = nn.Conv1d(input_dim, d_model, kernel_size=1)
        self.tcn = nn.ModuleList(
            [
                CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=1, dropout=dropout),
                CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=2, dropout=dropout),
                CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=4, dropout=dropout),
            ]
        )

        self.pos_emb = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.shape

        z = x.transpose(1, 2)
        z = self.input_proj(z)
        for block in self.tcn:
            z = block(z)
        z = z.transpose(1, 2)

        if self.pos_emb.size(1) >= seq_len:
            pos = self.pos_emb[:, :seq_len]
        else:
            pos = F.interpolate(
                self.pos_emb.transpose(1, 2),
                size=seq_len,
                mode='linear',
                align_corners=False,
            ).transpose(1, 2)

        z = z + pos
        attn_mask = torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), device=z.device),
            diagonal=1,
        )
        z = self.encoder(z, mask=attn_mask)
        z = self.norm(z[:, -1, :])

        out = self.head(z)
        out = out.view(bsz, self.horizon_count, self.num_classes)
        return out


class InceptionBlock1D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        b1 = out_ch // 4
        b2 = out_ch // 4
        b3 = out_ch // 4
        b4 = out_ch - (b1 + b2 + b3)

        self.branch1 = nn.Sequential(
            nn.Conv1d(in_ch, b1, kernel_size=1),
            nn.BatchNorm1d(b1),
            nn.GELU(),
        )
        self.branch2 = nn.Sequential(
            nn.Conv1d(in_ch, b2, kernel_size=1),
            nn.BatchNorm1d(b2),
            nn.GELU(),
            nn.Conv1d(b2, b2, kernel_size=3, padding=1),
            nn.BatchNorm1d(b2),
            nn.GELU(),
        )
        self.branch3 = nn.Sequential(
            nn.Conv1d(in_ch, b3, kernel_size=1),
            nn.BatchNorm1d(b3),
            nn.GELU(),
            nn.Conv1d(b3, b3, kernel_size=5, padding=2),
            nn.BatchNorm1d(b3),
            nn.GELU(),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_ch, b4, kernel_size=1),
            nn.BatchNorm1d(b4),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x),
        ], dim=1)


class HybridCNNInceptionLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int,
        horizon_count: int,
        num_classes: int = 3,
        channels: int = 96,
        lstm_hidden: int = 96,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes = num_classes

        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.GELU(),
        )
        self.inception = InceptionBlock1D(channels, channels)
        self.lstm = nn.LSTM(
            input_size=channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)
        z = x.transpose(1, 2)
        z = self.stem(z)
        z = self.inception(z)
        z = z.transpose(1, 2)
        z, _ = self.lstm(z)
        z = self.dropout(z[:, -1, :])

        out = self.head(z)
        out = out.view(bsz, self.horizon_count, self.num_classes)
        return out


class DeepLOBEncoder(nn.Module):
    def __init__(self, conv_channels: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(1, 2), stride=(1, 2)),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 32, kernel_size=(3, 1), padding=(1, 0)),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, conv_channels, kernel_size=(3, 1), padding=(1, 0)),
            nn.BatchNorm2d(conv_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, F]
        z = x.unsqueeze(1)
        z = self.conv(z)
        z = z.mean(dim=3)
        z = z.transpose(1, 2)
        return z


class Seq2SeqDeepLOBAttention(nn.Module):
    def __init__(
        self,
        input_dim: int,
        horizon_count: int,
        num_classes: int = 3,
        hidden_dim: int = 96,
        embed_dim: int = 16,
    ):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes = num_classes

        self.encoder_cnn = DeepLOBEncoder(conv_channels=64)
        self.encoder_lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
        )

        self.class_embed = nn.Embedding(num_classes, embed_dim)
        self.decoder_cell = nn.LSTMCell(hidden_dim + embed_dim, hidden_dim)
        self.query_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def _attend(self, enc_out: torch.Tensor, h_t: torch.Tensor) -> torch.Tensor:
        # enc_out: [B, T, H], h_t: [B, H]
        q = self.query_proj(h_t).unsqueeze(2)
        score = torch.bmm(enc_out, q).squeeze(2)
        alpha = torch.softmax(score, dim=1)
        ctx = torch.bmm(alpha.unsqueeze(1), enc_out).squeeze(1)
        return ctx

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)

        enc_in = self.encoder_cnn(x)
        enc_out, (h_n, c_n) = self.encoder_lstm(enc_in)

        h_t = h_n[-1]
        c_t = c_n[-1]

        prev_cls = torch.ones(bsz, dtype=torch.long, device=x.device)
        logits_steps = []

        for _ in range(self.horizon_count):
            cls_vec = self.class_embed(prev_cls)
            ctx = self._attend(enc_out, h_t)
            dec_in = torch.cat([ctx, cls_vec], dim=1)
            h_t, c_t = self.decoder_cell(dec_in, (h_t, c_t))

            step_logits = self.classifier(torch.cat([h_t, ctx], dim=1))
            logits_steps.append(step_logits)
            prev_cls = step_logits.argmax(dim=1)

        return torch.stack(logits_steps, dim=1)


def build_deep_model(arch: str, input_dim: int, horizon_count: int, num_classes: int = 3) -> nn.Module:
    if arch == 'dilated_transformer':
        return DilatedMaskedTransformer(
            input_dim=input_dim,
            horizon_count=horizon_count,
            num_classes=num_classes,
            d_model=96,
            n_heads=4,
            n_layers=2,
            dropout=0.15,
        )

    if arch == 'hybrid_cnn_inception_lstm':
        return HybridCNNInceptionLSTM(
            input_dim=input_dim,
            horizon_count=horizon_count,
            num_classes=num_classes,
            channels=96,
            lstm_hidden=96,
            dropout=0.2,
        )

    if arch == 'seq2seq_attn':
        return Seq2SeqDeepLOBAttention(
            input_dim=input_dim,
            horizon_count=horizon_count,
            num_classes=num_classes,
            hidden_dim=96,
            embed_dim=16,
        )

    raise ValueError(f'Unknown architecture: {arch}')


print('Cell 8 ready: deep model classes loaded.')


In [ ]:
# Stable data/loss helpers for the multi-epoch early-stop runner
if 'DEEP_CONFIG' in globals():
    DEEP_CONFIG.update({
        'max_files_per_ticker': 0,
        'result_suffix': '_stable_es',
        'label_smoothing': 0.02,
    })

class StableDayWindowDataset(Dataset):
    """Day dataset with per-day z-score normalization to prevent NaN blow-ups."""

    def __init__(self, raw: np.ndarray, starts: np.ndarray, labels: np.ndarray, seq_len: int):
        raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

        self.raw = raw
        self.starts = starts.astype(np.int64, copy=False)
        self.labels = labels.astype(np.int64, copy=False)
        self.seq_len = int(seq_len)

        feat_mean = raw.mean(axis=0, dtype=np.float64).astype(np.float32)
        feat_std = raw.std(axis=0, dtype=np.float64).astype(np.float32)
        feat_std = np.where(feat_std < 1e-6, 1.0, feat_std).astype(np.float32)

        self.feat_mean = feat_mean
        self.feat_std = feat_std

    def __len__(self) -> int:
        return int(self.starts.size)

    def __getitem__(self, idx: int):
        s = int(self.starts[idx])
        x = self.raw[s : s + self.seq_len]

        x = (x - self.feat_mean) / self.feat_std
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        x = np.clip(x, -10.0, 10.0)

        y = self.labels[idx]
        return torch.from_numpy(x.astype(np.float32, copy=False)), torch.from_numpy(y)


def _deep_build_day_dataset_stable(
    parquet_path: str,
    cfg: dict,
    is_train: bool,
) -> Tuple[StableDayWindowDataset | None, DayStats | None]:
    if 'make_fixed_threshold_classification_labels' not in globals():
        raise RuntimeError('make_fixed_threshold_classification_labels is not defined. Run labels cell first.')

    horizons = list(cfg['horizons'])
    seq_len = int(cfg['seq_len'])
    alpha = float(cfg['alpha'])

    step = _deep_choose_subsample(cfg)
    max_rows = _deep_choose_max_rows(cfg, is_train=is_train)

    df = pd.read_parquet(parquet_path, columns=DEEP_RAW_LOB_10_COLS)
    raw = np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=False))
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)

    y_float = make_fixed_threshold_classification_labels(
        df,
        horizons=horizons,
        alpha=alpha,
        use_smoothing=True,
    ).to_numpy(dtype=np.float32, copy=False)

    max_h = int(max(horizons))
    valid_end = len(df) - max_h

    if valid_end <= seq_len:
        del df, raw, y_float
        gc.collect()
        return None, None

    labels = y_float[seq_len - 1 : valid_end]
    valid_mask = ~np.isnan(labels).any(axis=1)
    starts = np.flatnonzero(valid_mask).astype(np.int64, copy=False)

    if step > 1:
        starts = starts[::step]

    if max_rows > 0:
        starts = starts[:max_rows]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts
        gc.collect()
        return None, None

    y_day = labels[starts].astype(np.int64, copy=False)
    class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
    starts = starts[class_mask]
    y_day = y_day[class_mask]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts, y_day, class_mask
        gc.collect()
        return None, None

    ds = StableDayWindowDataset(raw=raw, starts=starts, labels=y_day, seq_len=seq_len)
    stats = DayStats(
        ticker=os.path.basename(os.path.dirname(parquet_path)),
        file_name=os.path.basename(parquet_path),
        rows_raw=int(len(df)),
        rows_kept=int(starts.size),
        step=int(step),
        max_rows=int(max_rows),
    )

    del df, y_float, labels, valid_mask, starts, y_day, class_mask
    gc.collect()
    return ds, stats


def _deep_class_weights_stable(labels: np.ndarray) -> List[np.ndarray]:
    weights: List[np.ndarray] = []
    for i in range(labels.shape[1]):
        y = labels[:, i]
        # Add pseudocounts to avoid unstable swings when a class is sparse in a single day.
        counts = np.bincount(y.astype(np.int64, copy=False), minlength=3).astype(np.float64) + 10.0
        w = counts.sum() / (3.0 * counts)
        w = np.clip(w, 0.5, 3.0)
        weights.append(w.astype(np.float32))
    return weights


def _deep_multihorizon_loss_stable(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weight_tensors: List[torch.Tensor] | None = None,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    losses = []
    for i in range(logits.shape[1]):
        w = None if weight_tensors is None else weight_tensors[i]
        losses.append(
            F.cross_entropy(
                logits[:, i, :],
                targets[:, i],
                weight=w,
                label_smoothing=float(label_smoothing),
            )
        )
    return torch.stack(losses).mean()


In [ ]:
# Class imbalance handling utilities.
# Methods:
# - Class-Balanced Loss based on Effective Number of Samples (Cui et al., CVPR 2019):
#   https://arxiv.org/abs/1901.05555
# - Focal Loss for Dense Object Detection (Lin et al., ICCV 2017):
#   https://arxiv.org/abs/1708.02002

if "DEEP_CONFIG" in globals():
    DEEP_CONFIG.setdefault("loss_mode", "cb_focal")  # "ce" or "cb_focal"
    DEEP_CONFIG.setdefault("cb_beta", 0.999)
    DEEP_CONFIG.setdefault("cb_min_w", 0.5)
    DEEP_CONFIG.setdefault("cb_max_w", 3.0)
    DEEP_CONFIG.setdefault("cb_eps", 1.0)
    DEEP_CONFIG.setdefault("focal_gamma", 2.0)


def _deep_class_weights_stable(labels: np.ndarray) -> List[np.ndarray]:
    weights: List[np.ndarray] = []
    beta = float(globals().get("DEEP_CONFIG", {}).get("cb_beta", 0.999))
    min_w = float(globals().get("DEEP_CONFIG", {}).get("cb_min_w", 0.5))
    max_w = float(globals().get("DEEP_CONFIG", {}).get("cb_max_w", 3.0))
    eps = float(globals().get("DEEP_CONFIG", {}).get("cb_eps", 1.0))

    for i in range(labels.shape[1]):
        y = labels[:, i]
        counts = np.bincount(y.astype(np.int64, copy=False), minlength=3).astype(np.float64) + eps
        effective_num = 1.0 - np.power(beta, counts)
        w = (1.0 - beta) / np.maximum(effective_num, 1e-12)
        w = w / np.mean(w)
        w = np.clip(w, min_w, max_w)
        weights.append(w.astype(np.float32))
    return weights


def _deep_focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weight: torch.Tensor | None = None,
    gamma: float = 2.0,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    ce = F.cross_entropy(
        logits,
        targets,
        weight=weight,
        reduction="none",
        label_smoothing=float(label_smoothing),
    )
    p_t = torch.exp(-ce)
    loss = ((1.0 - p_t) ** float(gamma)) * ce
    return loss.mean()


def _deep_multihorizon_loss_stable(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weight_tensors: List[torch.Tensor] | None = None,
    label_smoothing: float = 0.0,
    loss_mode: str | None = None,
    focal_gamma: float | None = None,
) -> torch.Tensor:
    if loss_mode is None:
        loss_mode = str(globals().get("DEEP_CONFIG", {}).get("loss_mode", "ce"))
    if focal_gamma is None:
        focal_gamma = float(globals().get("DEEP_CONFIG", {}).get("focal_gamma", 2.0))

    losses = []
    for i in range(logits.shape[1]):
        w = None if weight_tensors is None else weight_tensors[i]
        if loss_mode == "cb_focal":
            loss_i = _deep_focal_loss(
                logits[:, i, :],
                targets[:, i],
                weight=w,
                gamma=float(focal_gamma),
                label_smoothing=float(label_smoothing),
            )
        else:
            loss_i = F.cross_entropy(
                logits[:, i, :],
                targets[:, i],
                weight=w,
                label_smoothing=float(label_smoothing),
            )
        losses.append(loss_i)
    return torch.stack(losses).mean()


def _deep_imbalance_summary(class_counts: dict[int, np.ndarray]) -> dict:
    summary: dict = {}
    for h, counts in class_counts.items():
        counts = counts.astype(np.float64, copy=False)
        total = float(counts.sum())
        if total <= 0:
            summary[f"h{h}"] = {
                "counts": [0, 0, 0],
                "percent": [0.0, 0.0, 0.0],
                "majority_pct": 0.0,
                "imbalance_ratio": float("inf"),
            }
            continue

        pct = counts / total
        nonzero = counts[counts > 0]
        min_count = float(nonzero.min()) if nonzero.size else 0.0
        max_count = float(counts.max()) if total > 0 else 0.0
        ratio = (max_count / min_count) if min_count > 0 else float("inf")

        summary[f"h{h}"] = {
            "counts": [int(x) for x in counts.tolist()],
            "percent": [round(float(p) * 100.0, 3) for p in pct.tolist()],
            "majority_pct": round(float(pct.max()) * 100.0, 3),
            "imbalance_ratio": round(float(ratio), 3) if math.isfinite(ratio) else float("inf"),
        }
    return summary


def _deep_print_imbalance_summary(title: str, summary: dict) -> None:
    print(f"\n{title} class imbalance:")
    for key in sorted(summary.keys(), key=lambda x: int(x[1:]) if x[1:].isdigit() else x):
        info = summary[key]
        print(
            f"  {key}: counts={info.get('counts')} percent={info.get('percent')} "
            f"majority%={info.get('majority_pct')} ratio={info.get('imbalance_ratio')}"
        )

print("Class imbalance helpers ready.")

In [ ]:
def _mean_macro_f1(metrics: dict, horizons: list[int]) -> float:
    vals = [float(metrics.get(f'h{h}_f1_macro', 0.0)) for h in horizons]
    return float(np.mean(vals)) if vals else 0.0


def _deep_train_epoch_stable(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
f    scaler: torch.amp.GradScaler,
    cfg: dict,
    train_files: list[tuple[str, str]],
    horizons: list[int],
    epoch_idx: int,
    total_epochs: int,
    amp_enabled: bool,
    label_smoothing: float,
    grad_clip: float,
    arch: str,
    start_file_idx: int = 1,
    resume_state: dict | None = None,
    checkpoint_callback = None,
) -> dict:
    model.train()
    print(f"\n[{arch}] epoch {epoch_idx}/{total_epochs}")

    if resume_state:
        train_rows_seen = resume_state['train_rows_seen']
        train_class_counts = resume_state['train_class_counts']
        bad_batches_total = resume_state['bad_batches']
        fitted_days = resume_state['fitted_days']
        day_losses = resume_state['day_losses']
    else:
        train_rows_seen = {h: 0 for h in horizons}
        train_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}
        bad_batches_total = 0
        fitted_days = 0
        day_losses: list[float] = []

    for file_idx, (ticker, path) in enumerate(train_files, start=1):
        if file_idx < start_file_idx:
            continue
        ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=True)
        if ds is None:
            print(f"[{arch}][train {file_idx}/{len(train_files)}] skipped empty day {os.path.basename(path)}")
            continue

        loader = _deep_make_loader(ds, cfg, is_train=True)
        day_weights_np = _deep_class_weights_stable(ds.labels)
        day_weights = [torch.tensor(w, dtype=torch.float32, device=DEEP_DEVICE) for w in day_weights_np]

        loss_sum = 0.0
        good_batches = 0
        bad_batches = 0

        for xb, yb in loader:
            xb = xb.to(DEEP_DEVICE, non_blocking=True)
            yb = yb.to(DEEP_DEVICE, non_blocking=True, dtype=torch.long)

            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)
            optimizer.zero_grad(set_to_none=True)

            if amp_enabled:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(xb)
                    logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                    loss = _deep_multihorizon_loss_stable(
                        logits,
                        yb,
                        day_weights,
                        label_smoothing=label_smoothing,
                    )

                if not torch.isfinite(loss):
                    bad_batches += 1
                    del xb, yb, logits, loss
                    continue

                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(xb)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                loss = _deep_multihorizon_loss_stable(
                    logits,
                    yb,
                    day_weights,
                    label_smoothing=label_smoothing,
                )

                if not torch.isfinite(loss):
                    bad_batches += 1
                    del xb, yb, logits, loss
                    continue

                loss.backward()
                if grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            loss_sum += float(loss.item())
            good_batches += 1

            del xb, yb, logits, loss

        bad_batches_total += bad_batches

        if good_batches == 0:
            print(
                f"[{arch}][train {file_idx}/{len(train_files)}] "
                f"{ticker}/{stats.file_name} had no finite batches; day skipped"
            )
            del loader, ds, day_weights, day_weights_np
            _deep_cleanup_cuda()
            continue

        fitted_days += 1
        day_loss = loss_sum / max(1, good_batches)
        day_losses.append(day_loss)

        for i, h in enumerate(horizons):
            train_rows_seen[h] += int(ds.labels.shape[0])
            train_class_counts[h] += np.bincount(ds.labels[:, i], minlength=3)

        print(
            f"[{arch}][train {file_idx}/{len(train_files)}] "
            f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
            f"max_rows={stats.max_rows} loss={day_loss:.4f} "
            f"good_batches={good_batches} bad_batches={bad_batches}"
        )

        del loader, ds, day_weights, day_weights_np
        _deep_cleanup_cuda()
        
        if checkpoint_callback is not None:
            checkpoint_callback(
                epoch=epoch_idx,
                file_idx=file_idx,
                epoch_state={
                    'train_rows_seen': train_rows_seen,
                    'train_class_counts': train_class_counts,
                    'bad_batches': bad_batches_total,
                    'fitted_days': fitted_days,
                    'day_losses': day_losses,
                }
            )

    return {
        'train_rows_seen': train_rows_seen,
        'train_class_counts': train_class_counts,
        'bad_batches': int(bad_batches_total),
        'fitted_days': int(fitted_days),
        'avg_train_loss': float(np.mean(day_losses)) if day_losses else float('nan'),
    }


def _deep_eval_epoch_stable(
    model: nn.Module,
    cfg: dict,
    eval_files: list[tuple[str, str]],
    horizons: list[int],
    arch: str,
) -> dict:
    confusion = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    eval_rows_seen = {h: 0 for h in horizons}
    eval_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    model.eval()
    with torch.no_grad():
        for file_idx, (ticker, path) in enumerate(eval_files, start=1):
            ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=False)
            if ds is None:
                print(f"[{arch}][eval {file_idx}/{len(eval_files)}] skipped empty day {os.path.basename(path)}")
                continue

            loader = _deep_make_loader(ds, cfg, is_train=False)

            for xb, yb in loader:
                xb_gpu = xb.to(DEEP_DEVICE, non_blocking=True)
                xb_gpu = torch.nan_to_num(xb_gpu, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)

                logits = model(xb_gpu)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)

                preds = logits.argmax(dim=2).detach().cpu().numpy().astype(np.int64, copy=False)
                y_true = yb.numpy().astype(np.int64, copy=False)

                batch_n = int(y_true.shape[0])
                for i, h in enumerate(horizons):
                    _deep_update_confusion(confusion[h], y_true[:, i], preds[:, i])
                    eval_rows_seen[h] += batch_n
                    eval_class_counts[h] += np.bincount(y_true[:, i], minlength=3)

                del xb, xb_gpu, yb, logits, preds, y_true

            print(
                f"[{arch}][eval {file_idx}/{len(eval_files)}] "
                f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
                f"max_rows={stats.max_rows}"
            )

            del loader, ds
            _deep_cleanup_cuda()

    metrics: Dict[str, float] = {}
    for h in horizons:
        metrics.update(_deep_metrics_from_confusion(confusion[h], h))

    return {
        'metrics': metrics,
        'confusion': confusion,
        'eval_rows_seen': eval_rows_seen,
        'eval_class_counts': eval_class_counts,
    }


def train_one_deep_architecture_stable_earlystop(
    arch: str,
    config: dict | None = None,
    max_epochs: int = 6,
    patience: int = 2,
    min_delta: float = 1e-4,
) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    cfg['max_files_per_ticker'] = 0
    cfg['epochs'] = 1

    horizons = list(cfg['horizons'])
    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    tickers = _deep_resolve_tickers(cfg)
    if not tickers:
        raise FileNotFoundError(f"No tickers found in {cfg['data_dir']}")

    files_by_ticker = _deep_collect_files_by_ticker(
        cfg['data_dir'],
        tickers,
        int(cfg['max_files_per_ticker']),
    )
    if not files_by_ticker:
        raise FileNotFoundError('No parquet files found for selected tickers.')

    train_files, eval_files = _deep_split_train_eval_files(
        files_by_ticker,
        float(cfg['train_file_fraction']),
    )

    model = build_deep_model(
        arch=arch,
        input_dim=len(DEEP_RAW_LOB_10_COLS),
        horizon_count=len(horizons),
        num_classes=3,
    ).to(DEEP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg['lr']),
        weight_decay=float(cfg['weight_decay']),
    )

    amp_enabled = bool(cfg.get('amp', False)) and DEEP_DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    grad_clip = float(cfg.get('grad_clip', 1.0))
    label_smoothing = float(cfg.get('label_smoothing', 0.0))

    start_t = time.time()
    best_score = -1.0
    best_epoch = 0
    best_metrics: dict = {}
    best_eval_rows_seen = {h: 0 for h in horizons}
    best_eval_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}
    best_state_dict: dict | None = None

    no_improve = 0
    epoch_history: list[dict] = []

    total_bad_batches = 0
    total_train_days = 0
    agg_train_rows_seen = {h: 0 for h in horizons}
    agg_train_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    start_epoch = 1
    start_file_idx = 1
    resume_epoch_state = None
    checkpoint_path = os.path.join(cfg['weights_dir'], f'{arch}_file_checkpoint.pt')

    if os.path.exists(checkpoint_path):
        print(f"[{arch}] Found existing checkpoint at {checkpoint_path}, resuming...")
        chk = torch.load(checkpoint_path, map_location=DEEP_DEVICE, weights_only=False)
        model.load_state_dict(chk['model'])
        optimizer.load_state_dict(chk['optimizer'])
        if scaler and chk.get('scaler'):
            scaler.load_state_dict(chk['scaler'])
        
        start_epoch = int(chk['epoch'])
        start_file_idx = int(chk['file_idx']) + 1
        if start_file_idx > len(train_files):
            start_epoch += 1
            start_file_idx = 1
            resume_epoch_state = None
        else:
            resume_epoch_state = chk.get('epoch_state')

        best_score = float(chk['best_score'])
        best_epoch = int(chk['best_epoch'])
        best_metrics = chk.get('best_metrics', {})
        best_eval_rows_seen = chk.get('best_eval_rows_seen', best_eval_rows_seen)
        best_eval_class_counts = chk.get('best_eval_class_counts', best_eval_class_counts)
        best_state_dict = chk.get('best_state_dict')
        no_improve = int(chk.get('no_improve', 0))
        epoch_history = chk.get('epoch_history', [])
        total_bad_batches = int(chk.get('total_bad_batches', 0))
        total_train_days = int(chk.get('total_train_days', 0))
        agg_train_rows_seen = chk.get('agg_train_rows_seen', agg_train_rows_seen)
        agg_train_class_counts = chk.get('agg_train_class_counts', agg_train_class_counts)

    def save_file_checkpoint(epoch: int, file_idx: int, epoch_state: dict):
        chk_data = {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scaler': scaler.state_dict() if scaler else None,
            'epoch': epoch,
            'file_idx': file_idx,
            'epoch_state': epoch_state,
            'best_score': best_score,
            'best_epoch': best_epoch,
            'best_metrics': best_metrics,
            'best_eval_rows_seen': best_eval_rows_seen,
            'best_eval_class_counts': best_eval_class_counts,
            'best_state_dict': {k: v.cpu() for k, v in best_state_dict.items()} if best_state_dict else None,
            'no_improve': no_improve,
            'epoch_history': epoch_history,
            'total_bad_batches': total_bad_batches,
            'total_train_days': total_train_days,
            'agg_train_rows_seen': agg_train_rows_seen,
            'agg_train_class_counts': agg_train_class_counts,
        }
        torch.save(chk_data, checkpoint_path)

    print('\n' + '=' * 88)
    print(f"Stable early-stop training architecture: {arch}")
    print(
        f"Device={DEEP_DEVICE} | train_files={len(train_files)} eval_files={len(eval_files)} "
        f"| max_epochs={max_epochs} patience={patience}"
    )

    for epoch in range(start_epoch, int(max_epochs) + 1):
        train_info = _deep_train_epoch_stable(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            cfg=cfg,
            train_files=train_files,
            horizons=horizons,
            epoch_idx=epoch,
            total_epochs=int(max_epochs),
            amp_enabled=amp_enabled,
            label_smoothing=label_smoothing,
            grad_clip=grad_clip,
            arch=arch,
            start_file_idx=start_file_idx,
            resume_state=resume_epoch_state,
            checkpoint_callback=save_file_checkpoint,
        )
        start_file_idx = 1
        resume_epoch_state = None

        total_bad_batches += int(train_info['bad_batches'])
        total_train_days += int(train_info['fitted_days'])
        for h in horizons:
            agg_train_rows_seen[h] += int(train_info['train_rows_seen'][h])
            agg_train_class_counts[h] += train_info['train_class_counts'][h]

        eval_info = _deep_eval_epoch_stable(
            model=model,
            cfg=cfg,
            eval_files=eval_files,
            horizons=horizons,
            arch=arch,
        )

        metrics = eval_info['metrics']
        score = _mean_macro_f1(metrics, horizons)
        improved = score > (best_score + float(min_delta))

        epoch_history.append({
            'epoch': int(epoch),
            'mean_macro_f1': float(score),
            'avg_train_loss': float(train_info['avg_train_loss']),
            'bad_batches': int(train_info['bad_batches']),
            'improved': bool(improved),
        })

        print(
            f"[{arch}] epoch={epoch} mean_macro_f1={score:.6f} "
            f"best={best_score:.6f} improved={improved}"
        )

        if improved:
            best_score = float(score)
            best_epoch = int(epoch)
            best_metrics = dict(metrics)
            best_eval_rows_seen = {h: int(eval_info['eval_rows_seen'][h]) for h in horizons}
            best_eval_class_counts = {h: eval_info['eval_class_counts'][h].copy() for h in horizons}
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= int(patience):
            print(f"[{arch}] early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

        _deep_cleanup_cuda()

    if best_state_dict is None:
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = int(len(epoch_history)) if epoch_history else 1
        best_metrics = dict(eval_info['metrics']) if 'eval_info' in locals() else {}
        best_score = _mean_macro_f1(best_metrics, horizons)

    model.load_state_dict(best_state_dict)

    suffix = str(cfg.get('result_suffix', '_stable_es'))
    if suffix and not suffix.startswith('_'):
        suffix = '_' + suffix

    weights_path = os.path.join(cfg['weights_dir'], f'{arch}{suffix}_weights.pt')
    torch.save(
        {
            'architecture': arch,
            'state_dict': model.state_dict(),
            'input_dim': len(DEEP_RAW_LOB_10_COLS),
            'horizons': horizons,
            'seq_len': int(cfg['seq_len']),
            'alpha': float(cfg['alpha']),
            'device_trained': str(DEEP_DEVICE),
            'stable_training': True,
            'early_stopping': {
                'max_epochs': int(max_epochs),
                'patience': int(patience),
                'min_delta': float(min_delta),
                'best_epoch': int(best_epoch),
                'best_mean_macro_f1': float(best_score),
                'epochs_ran': int(len(epoch_history)),
            },
        },
        weights_path,
    )

    runtime_s = time.time() - start_t
    run_meta = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep_stable_earlystop',
        'architecture': arch,
        'tickers': tickers,
        'horizons': horizons,
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'train_config': {
            'max_epochs': int(max_epochs),
            'patience': int(patience),
            'min_delta': float(min_delta),
            'batch_size': int(cfg['batch_size']),
            'lr': float(cfg['lr']),
            'weight_decay': float(cfg['weight_decay']),
            'grad_clip': float(cfg['grad_clip']),
            'amp': bool(cfg.get('amp', False)),
            'label_smoothing': float(cfg.get('label_smoothing', 0.0)),
            'device': str(DEEP_DEVICE),
        },
        'sampling': {
            'train_file_fraction': float(cfg['train_file_fraction']),
            'base_subsample': int(cfg['base_subsample']),
            'high_pressure_subsample': int(cfg['high_pressure_subsample']),
            'critical_pressure_subsample': int(cfg['critical_pressure_subsample']),
            'max_rows_per_day_train': int(cfg['max_rows_per_day_train']),
            'max_rows_per_day_eval': int(cfg['max_rows_per_day_eval']),
            'max_files_per_ticker': int(cfg['max_files_per_ticker']),
        },
        'files': {
            'train_files': len(train_files),
            'eval_files': len(eval_files),
            'days_fitted': int(total_train_days),
        },
        'early_stopping': {
            'best_epoch': int(best_epoch),
            'best_mean_macro_f1': float(best_score),
            'epochs_ran': int(len(epoch_history)),
        },
        'epoch_history': epoch_history,
        'train_rows_seen': {f'h{h}': int(agg_train_rows_seen[h]) for h in horizons},
        'eval_rows_seen': {f'h{h}': int(best_eval_rows_seen[h]) for h in horizons},
        'train_class_counts': {f'h{h}': [int(x) for x in agg_train_class_counts[h].tolist()] for h in horizons},
        'eval_class_counts': {f'h{h}': [int(x) for x in best_eval_class_counts[h].tolist()] for h in horizons},
        'bad_batches_total': int(total_bad_batches),
        'test_metrics': best_metrics,
        'model_path': weights_path,
        'runtime_seconds': round(runtime_s, 2),
    }

    results_path = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming{suffix}.json')
    with open(results_path, 'w') as f:
        json.dump(run_meta, f, indent=2)

    print(f"[{arch}] stable-es saved weights -> {weights_path}")
    print(f"[{arch}] stable-es saved metrics -> {results_path}")

    del model, optimizer, scaler
    _deep_cleanup_cuda()

    return run_meta


def run_all_deep_models_day_by_day_stable_earlystop(
    config: dict | None = None,
    max_epochs: int = 6,
    patience: int = 2,
    min_delta: float = 1e-4,
) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    cfg['max_files_per_ticker'] = 0
    cfg.setdefault('result_suffix', '_stable_es')

    deep_set_seed(int(cfg['seed']))
    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    all_runs: Dict[str, dict] = {}
    all_results_paths: Dict[str, str] = {}

    t0 = time.time()
    for arch in cfg['run_architectures']:
        run_meta = train_one_deep_architecture_stable_earlystop(
            arch=arch,
            config=cfg,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
        )
        all_runs[arch] = run_meta

        suffix = str(cfg.get('result_suffix', '_stable_es'))
        if suffix and not suffix.startswith('_'):
            suffix = '_' + suffix
        all_results_paths[arch] = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming{suffix}.json')

        _deep_cleanup_cuda()

    suffix = str(cfg.get('result_suffix', '_stable_es'))
    if suffix and not suffix.startswith('_'):
        suffix = '_' + suffix

    summary = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep_multi_arch_stable_earlystop',
        'architectures': list(cfg['run_architectures']),
        'horizons': list(cfg['horizons']),
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'early_stopping': {
            'max_epochs': int(max_epochs),
            'patience': int(patience),
            'min_delta': float(min_delta),
        },
        'results_paths': all_results_paths,
        'runtime_seconds': round(time.time() - t0, 2),
    }

    summary_path = os.path.join(cfg['results_dir'], f'deep_models_day_streaming_summary{suffix}.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print('\n' + '=' * 88)
    print('Stable deep early-stop run complete')
    print(f"Summary saved -> {summary_path}")

    return {
        'summary_path': summary_path,
        'summary': summary,
        'runs': all_runs,
    }

# Execute multi-epoch deep training (all files, day-by-day streaming).
DEEP_EPOCH_RUN_CONFIG = {
    'result_suffix': '_stable_es',
    'max_files_per_ticker': 0,
    'amp': bool(DEEP_CONFIG.get('amp', True)),
    'label_smoothing': float(DEEP_CONFIG.get('label_smoothing', 0.02)),
}
DEEP_MAX_EPOCHS = 6
DEEP_EARLY_STOP_PATIENCE = 2
DEEP_EARLY_STOP_MIN_DELTA = 1e-4

deep_runs_stable_es = run_all_deep_models_day_by_day_stable_earlystop(
    config=DEEP_EPOCH_RUN_CONFIG,
    max_epochs=DEEP_MAX_EPOCHS,
    patience=DEEP_EARLY_STOP_PATIENCE,
    min_delta=DEEP_EARLY_STOP_MIN_DELTA,
)

for arch, run_meta in deep_runs_stable_es['runs'].items():
    deep_score = _mean_macro_f1(run_meta.get('test_metrics', {}), list(DEEP_CONFIG['horizons']))
    best_epoch = int(run_meta.get('early_stopping', {}).get('best_epoch', 0))
    print(f"{arch}: mean macro-F1={deep_score:.6f} | best_epoch={best_epoch}")

print(f"Deep early-stop summary: {deep_runs_stable_es['summary_path']}")


In [ ]:
def _deep_counts_from_meta(meta_counts: dict) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for key, values in meta_counts.items():
        if isinstance(key, str) and key.startswith("h"):
            h = int(key[1:])
        else:
            h = int(key)
        out[h] = np.array(values, dtype=np.int64)
    return out


if "deep_runs_stable_es" in globals():
    runs = deep_runs_stable_es.get("runs", {})
    for arch, meta in runs.items():
        print(f"\n=== {arch} imbalance summary ===")
        train_counts = _deep_counts_from_meta(meta.get("train_class_counts", {}))
        eval_counts = _deep_counts_from_meta(meta.get("eval_class_counts", {}))

        if train_counts:
            _deep_print_imbalance_summary("Train", _deep_imbalance_summary(train_counts))
        if eval_counts:
            _deep_print_imbalance_summary("Eval", _deep_imbalance_summary(eval_counts))
else:
    print("Run deep training first to compute imbalance summaries.")

In [ ]:
# Balanced evaluation without retraining (random undersampling / stratified subsampling).
# Method references:
# - He and Garcia (2009) Learning from imbalanced data, IEEE TKDE.
#   https://ieeexplore.ieee.org/document/4781139
# - Buda, Maki, Mazurowski (2018) A systematic study of the class imbalance problem in CNNs.
#   https://arxiv.org/abs/1710.05381


def _deep_target_counts_from_meta(meta_counts: dict) -> dict[int, np.ndarray]:
    targets: dict[int, np.ndarray] = {}
    for key, values in meta_counts.items():
        if isinstance(key, str) and key.startswith("h"):
            h = int(key[1:])
        else:
            h = int(key)
        counts = np.array(values, dtype=np.int64)
        min_count = int(counts.min()) if counts.size else 0
        targets[h] = np.full(3, min_count, dtype=np.int64)
    return targets


def _deep_balanced_mask(
    y_true_h: np.ndarray,
    used_counts: np.ndarray,
    target_counts: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    mask = np.zeros(y_true_h.shape[0], dtype=bool)
    for cls in range(3):
        remaining = int(target_counts[cls] - used_counts[cls])
        if remaining <= 0:
            continue
        idx = np.flatnonzero(y_true_h == cls)
        if idx.size == 0:
            continue
        if idx.size > remaining:
            choose = rng.choice(idx, size=remaining, replace=False)
        else:
            choose = idx
        if choose.size:
            mask[choose] = True
            used_counts[cls] += int(choose.size)
    return mask


def _deep_balanced_done(used: dict[int, np.ndarray], target: dict[int, np.ndarray]) -> bool:
    for h, tgt in target.items():
        if h not in used:
            return False
        if np.any(used[h] < tgt):
            return False
    return True


def _deep_eval_balanced_undersample(
    model: nn.Module,
    cfg: dict,
    eval_files: list[tuple[str, str]],
    horizons: list[int],
    target_counts: dict[int, np.ndarray],
    seed: int,
) -> dict:
    confusion = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    used_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}
    rng = np.random.default_rng(int(seed))

    model.eval()
    with torch.no_grad():
        for file_idx, (ticker, path) in enumerate(eval_files, start=1):
            ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=False)
            if ds is None:
                continue

            loader = _deep_make_loader(ds, cfg, is_train=False)
            for xb, yb in loader:
                xb_gpu = xb.to(DEEP_DEVICE, non_blocking=True)
                xb_gpu = torch.nan_to_num(xb_gpu, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)

                logits = model(xb_gpu)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)

                preds = logits.argmax(dim=2).detach().cpu().numpy().astype(np.int64, copy=False)
                y_true = yb.numpy().astype(np.int64, copy=False)

                for i, h in enumerate(horizons):
                    target = target_counts.get(h)
                    if target is None or int(target.min()) <= 0:
                        continue
                    mask = _deep_balanced_mask(y_true[:, i], used_counts[h], target, rng)
                    if mask.any():
                        _deep_update_confusion(confusion[h], y_true[mask, i], preds[mask, i])

                del xb, yb, xb_gpu, logits, preds, y_true

            del loader, ds
            _deep_cleanup_cuda()

            if _deep_balanced_done(used_counts, target_counts):
                break

    metrics: dict[str, float] = {}
    for h in horizons:
        metrics.update(_deep_metrics_from_confusion(confusion[h], h))

    return {
        "metrics": metrics,
        "balanced_counts": {h: used_counts[h].copy() for h in horizons},
        "confusion": confusion,
    }


def _deep_load_saved_runs_from_results(cfg: dict) -> dict:
    runs: dict = {}
    suffix = str(cfg.get("result_suffix", "_stable_es"))
    if suffix and not suffix.startswith("_"):
        suffix = "_" + suffix
    for arch in cfg.get("run_architectures", []):
        path = os.path.join(cfg["results_dir"], f"{arch}_results_day_streaming{suffix}.json")
        if os.path.isfile(path):
            with open(path, "r") as f:
                runs[arch] = json.load(f)
    return runs


def _deep_compare_metrics(base_metrics: dict, balanced_metrics: dict, horizons: list[int]) -> dict:
    delta: dict[str, float] = {}
    keys = ["accuracy", "f1_macro", "f1_weighted", "precision_macro", "recall_macro"]
    for h in horizons:
        for key in keys:
            k = f"h{h}_{key}"
            base = float(base_metrics.get(k, 0.0))
            bal = float(balanced_metrics.get(k, 0.0))
            delta[k] = round(bal - base, 6)
    return delta


cfg = dict(DEEP_CONFIG)
horizons = list(cfg["horizons"])

base_runs: dict = {}
if "deep_runs_stable_es" in globals() and isinstance(deep_runs_stable_es, dict):
    base_runs = deep_runs_stable_es.get("runs", {})
if not base_runs:
    base_runs = _deep_load_saved_runs_from_results(cfg)

if not base_runs:
    print("No saved runs found. Run the training cell first.")
else:
    tickers = _deep_resolve_tickers(cfg)
    files_by_ticker = _deep_collect_files_by_ticker(
        cfg["data_dir"],
        tickers,
        int(cfg.get("max_files_per_ticker", 0)),
    )
    _, eval_files = _deep_split_train_eval_files(
        files_by_ticker,
        float(cfg.get("train_file_fraction", 0.8)),
    )

    balanced_runs: dict = {}
    comparison_runs: dict = {}

    for arch, meta in base_runs.items():
        weight_path = meta.get("model_path")
        if not weight_path:
            suffix = str(cfg.get("result_suffix", "_stable_es"))
            if suffix and not suffix.startswith("_"):
                suffix = "_" + suffix
            weight_path = os.path.join(cfg["weights_dir"], f"{arch}{suffix}_weights.pt")

        if not os.path.isfile(weight_path):
            print(f"[skip] weights not found for {arch}: {weight_path}")
            continue

        model = build_deep_model(
            arch=arch,
            input_dim=len(DEEP_RAW_LOB_10_COLS),
            horizon_count=len(horizons),
            num_classes=3,
        ).to(DEEP_DEVICE)

        state = torch.load(weight_path, map_location=DEEP_DEVICE, weights_only=False)
        state_dict = state.get("state_dict", state)
        model.load_state_dict(state_dict)

        target_counts = _deep_target_counts_from_meta(meta.get("eval_class_counts", {}))
        balanced_eval = _deep_eval_balanced_undersample(
            model=model,
            cfg=cfg,
            eval_files=eval_files,
            horizons=horizons,
            target_counts=target_counts,
            seed=int(cfg.get("seed", 42)),
        )

        balanced_counts = {f"h{h}": [int(x) for x in balanced_eval["balanced_counts"][h].tolist()] for h in horizons}
        target_counts_out = {f"h{h}": [int(x) for x in target_counts.get(h, np.zeros(3, dtype=np.int64)).tolist()] for h in horizons}

        balanced_runs[arch] = {
            "model_path": weight_path,
            "balanced_metrics": balanced_eval["metrics"],
            "balanced_counts": balanced_counts,
            "target_counts": target_counts_out,
            "eval_files": int(len(eval_files)),
        }

        comparison_runs[arch] = {
            "base_metrics": meta.get("test_metrics", {}),
            "balanced_metrics": balanced_eval["metrics"],
            "delta_metrics": _deep_compare_metrics(meta.get("test_metrics", {}), balanced_eval["metrics"], horizons),
        }

        print(f"\n=== {arch} balanced eval summary ===")
        _deep_print_imbalance_summary("Balanced", _deep_imbalance_summary({h: balanced_eval["balanced_counts"][h] for h in horizons}))

        del model
        _deep_cleanup_cuda()

    timestamp = pd.Timestamp.now().isoformat()
    suffix = str(cfg.get("result_suffix", "_stable_es"))
    if suffix and not suffix.startswith("_"):
        suffix = "_" + suffix

    balanced_payload = {
        "timestamp": timestamp,
        "mode": "balanced_eval_undersample_no_retrain",
        "method": {
            "name": "random_undersampling_stratified",
            "references": [
                "He and Garcia 2009 (IEEE TKDE)",
                "Buda et al. 2018 (arXiv:1710.05381)",
            ],
        },
        "horizons": horizons,
        "runs": balanced_runs,
    }

    comparison_payload = {
        "timestamp": timestamp,
        "mode": "comparison_balanced_vs_original",
        "horizons": horizons,
        "runs": comparison_runs,
    }

    balanced_path = os.path.join(cfg["results_dir"], f"deep_models_balanced_eval{suffix}.json")
    compare_path = os.path.join(cfg["results_dir"], f"deep_models_balanced_comparison{suffix}.json")

    with open(balanced_path, "w") as f:
        json.dump(balanced_payload, f, indent=2)
    with open(compare_path, "w") as f:
        json.dump(comparison_payload, f, indent=2)

    print(f"\nBalanced eval results saved -> {balanced_path}")
    print(f"Comparison results saved -> {compare_path}")

In [ ]:
# Balanced training via random undersampling (no retraining on imbalanced data).
# Method references:
# - He and Garcia (2009) Learning from imbalanced data, IEEE TKDE.
#   https://ieeexplore.ieee.org/document/4781139
# - Buda, Maki, Mazurowski (2018) A systematic study of the class imbalance problem in CNNs.
#   https://arxiv.org/abs/1710.05381

import hashlib


def _deep_seed_from_path(path: str, base_seed: int) -> int:
    digest = hashlib.md5(path.encode("utf-8")).hexdigest()
    return (int(digest[:8], 16) + int(base_seed)) % (2**32 - 1)


def _deep_build_day_dataset_balanced(
    parquet_path: str,
    cfg: dict,
    is_train: bool,
) -> Tuple[StableDayWindowDataset | None, DayStats | None]:
    if "make_fixed_threshold_classification_labels" not in globals():
        raise RuntimeError("make_fixed_threshold_classification_labels is not defined. Run labels cell first.")

    horizons = list(cfg["horizons"])
    seq_len = int(cfg["seq_len"])
    alpha = float(cfg["alpha"])

    step = _deep_choose_subsample(cfg)
    max_rows = _deep_choose_max_rows(cfg, is_train=is_train)

    df = pd.read_parquet(parquet_path, columns=DEEP_RAW_LOB_10_COLS)
    raw = np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=False))
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)

    y_float = make_fixed_threshold_classification_labels(
        df,
        horizons=horizons,
        alpha=alpha,
        use_smoothing=True,
    ).to_numpy(dtype=np.float32, copy=False)

    max_h = int(max(horizons))
    valid_end = len(df) - max_h

    if valid_end <= seq_len:
        del df, raw, y_float
        gc.collect()
        return None, None

    labels = y_float[seq_len - 1 : valid_end]
    valid_mask = ~np.isnan(labels).any(axis=1)
    starts = np.flatnonzero(valid_mask).astype(np.int64, copy=False)

    if step > 1:
        starts = starts[::step]

    if max_rows > 0:
        starts = starts[:max_rows]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts
        gc.collect()
        return None, None

    y_day = labels[starts].astype(np.int64, copy=False)
    class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
    starts = starts[class_mask]
    y_day = y_day[class_mask]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts, y_day, class_mask
        gc.collect()
        return None, None

    if is_train and str(cfg.get("balance_mode", "")) == "undersample":
        balance_h = int(cfg.get("balance_horizon", horizons[0]))
        try:
            h_idx = horizons.index(balance_h)
        except ValueError:
            h_idx = 0

        y_h = y_day[:, h_idx]
        counts = np.bincount(y_h, minlength=3).astype(np.int64)
        min_count = int(counts.min()) if counts.size else 0
        min_required = int(cfg.get("balance_min_per_class", 1))

        if min_count < min_required:
            del df, raw, y_float, labels, valid_mask, starts, y_day, class_mask
            gc.collect()
            return None, None

        seed = _deep_seed_from_path(parquet_path, int(cfg.get("seed", 42)))
        rng = np.random.default_rng(seed)

        idx = []
        for cls in range(3):
            cls_idx = np.flatnonzero(y_h == cls)
            choose = rng.choice(cls_idx, size=min_count, replace=False)
            idx.append(choose)
        idx = np.concatenate(idx)
        rng.shuffle(idx)

        starts = starts[idx]
        y_day = y_day[idx]

    ds = StableDayWindowDataset(raw=raw, starts=starts, labels=y_day, seq_len=seq_len)
    stats = DayStats(
        ticker=os.path.basename(os.path.dirname(parquet_path)),
        file_name=os.path.basename(parquet_path),
        rows_raw=int(len(df)),
        rows_kept=int(starts.size),
        step=int(step),
        max_rows=int(max_rows),
    )

    del df, y_float, labels, valid_mask, starts, y_day, class_mask
    gc.collect()
    return ds, stats


cfg_base = dict(DEEP_CONFIG)
base_suffix = str(cfg_base.get("result_suffix", "_stable_es"))

balanced_train_config = {
    "result_suffix": "_balanced_train_es",
    "balance_mode": "undersample",
    "balance_horizon": int(cfg_base.get("horizons", [10])[0]),
    "balance_min_per_class": 1,
}

balanced_epochs = int(globals().get("DEEP_MAX_EPOCHS", 6))
balanced_patience = int(globals().get("DEEP_EARLY_STOP_PATIENCE", 2))
balanced_min_delta = float(globals().get("DEEP_EARLY_STOP_MIN_DELTA", 1e-4))

orig_builder = _deep_build_day_dataset_stable
orig_weights_fn = _deep_class_weights_stable
orig_loss_mode = DEEP_CONFIG.get("loss_mode")


def _deep_class_weights_uniform(labels: np.ndarray) -> List[np.ndarray]:
    return [np.ones(3, dtype=np.float32) for _ in range(labels.shape[1])]

try:
    _deep_build_day_dataset_stable = _deep_build_day_dataset_balanced
    _deep_class_weights_stable = _deep_class_weights_uniform
    DEEP_CONFIG["loss_mode"] = "ce"

    balanced_runs_out = run_all_deep_models_day_by_day_stable_earlystop(
        config=balanced_train_config,
        max_epochs=balanced_epochs,
        patience=balanced_patience,
        min_delta=balanced_min_delta,
    )
finally:
    _deep_build_day_dataset_stable = orig_builder
    _deep_class_weights_stable = orig_weights_fn
    if orig_loss_mode is None:
        DEEP_CONFIG.pop("loss_mode", None)
    else:
        DEEP_CONFIG["loss_mode"] = orig_loss_mode

base_runs_saved = _deep_load_saved_runs_from_results({**cfg_base, "result_suffix": base_suffix})

balanced_runs = balanced_runs_out.get("runs", {}) if "balanced_runs_out" in globals() else {}
comparison_runs: dict = {}

if not base_runs_saved:
    print("No saved baseline runs found to compare against.")
else:
    for arch, meta in balanced_runs.items():
        base_meta = base_runs_saved.get(arch, {})
        comparison_runs[arch] = {
            "base_metrics": base_meta.get("test_metrics", {}),
            "balanced_metrics": meta.get("test_metrics", {}),
            "delta_metrics": _deep_compare_metrics(
                base_meta.get("test_metrics", {}),
                meta.get("test_metrics", {}),
                list(cfg_base.get("horizons", [])),
            ),
        }

    timestamp = pd.Timestamp.now().isoformat()
    balanced_suffix = str(balanced_train_config.get("result_suffix", "_balanced_train_es"))
    if balanced_suffix and not balanced_suffix.startswith("_"):
        balanced_suffix = "_" + balanced_suffix

    balanced_train_payload = {
        "timestamp": timestamp,
        "mode": "balanced_train_undersample",
        "method": {
            "name": "random_undersampling_stratified",
            "references": [
                "He and Garcia 2009 (IEEE TKDE)",
                "Buda et al. 2018 (arXiv:1710.05381)",
            ],
        },
        "horizons": list(cfg_base.get("horizons", [])),
        "summary_path": balanced_runs_out.get("summary_path") if "balanced_runs_out" in globals() else None,
        "runs": balanced_runs,
    }

    comparison_payload = {
        "timestamp": timestamp,
        "mode": "comparison_balanced_train_vs_original",
        "horizons": list(cfg_base.get("horizons", [])),
        "runs": comparison_runs,
    }

    balanced_train_path = os.path.join(cfg_base["results_dir"], f"deep_models_balanced_train_results{balanced_suffix}.json")
    compare_path = os.path.join(cfg_base["results_dir"], f"deep_models_balanced_train_comparison_vs_original{balanced_suffix}.json")

    with open(balanced_train_path, "w") as f:
        json.dump(balanced_train_payload, f, indent=2)
    with open(compare_path, "w") as f:
        json.dump(comparison_payload, f, indent=2)

    print(f"\nBalanced training results saved -> {balanced_train_path}")
    print(f"Comparison vs original saved -> {compare_path}")

In [ ]:
# SMOTE oversampling training + comparison vs imbalanced and undersampled.
# Method references:
# - Chawla et al. (2002) SMOTE: Synthetic Minority Over-sampling Technique.
#   https://jair.org/index.php/jair/article/view/10302
# - He and Garcia (2009) Learning from imbalanced data, IEEE TKDE.
#   https://ieeexplore.ieee.org/document/4781139


def _deep_build_day_dataset_smote(
    parquet_path: str,
    cfg: dict,
    is_train: bool,
) -> Tuple[Dataset | None, DayStats | None]:
    if "make_fixed_threshold_classification_labels" not in globals():
        raise RuntimeError("make_fixed_threshold_classification_labels is not defined. Run labels cell first.")

    horizons = list(cfg["horizons"])
    seq_len = int(cfg["seq_len"])
    alpha = float(cfg["alpha"])

    step = _deep_choose_subsample(cfg)
    max_rows = _deep_choose_max_rows(cfg, is_train=is_train)

    df = pd.read_parquet(parquet_path, columns=DEEP_RAW_LOB_10_COLS)
    raw = np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=False))
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)

    y_float = make_fixed_threshold_classification_labels(
        df,
        horizons=horizons,
        alpha=alpha,
        use_smoothing=True,
    ).to_numpy(dtype=np.float32, copy=False)

    max_h = int(max(horizons))
    valid_end = len(df) - max_h

    if valid_end <= seq_len:
        del df, raw, y_float
        gc.collect()
        return None, None

    labels = y_float[seq_len - 1 : valid_end]
    valid_mask = ~np.isnan(labels).any(axis=1)
    starts = np.flatnonzero(valid_mask).astype(np.int64, copy=False)

    if step > 1:
        starts = starts[::step]

    if max_rows > 0:
        starts = starts[:max_rows]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts
        gc.collect()
        return None, None

    y_day = labels[starts].astype(np.int64, copy=False)
    class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
    starts = starts[class_mask]
    y_day = y_day[class_mask]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts, y_day, class_mask
        gc.collect()
        return None, None

    synth_x = None
    synth_y = None

    if is_train and str(cfg.get("balance_mode", "")) == "smote":
        balance_h = int(cfg.get("balance_horizon", horizons[0]))
        try:
            h_idx = horizons.index(balance_h)
        except ValueError:
            h_idx = 0

        y_h = y_day[:, h_idx]
        counts = np.bincount(y_h, minlength=3).astype(np.int64)
        min_required = int(cfg.get("balance_min_per_class", 2))
        if int(counts.min()) < min_required:
            del df, raw, y_float, labels, valid_mask, starts, y_day, class_mask
            gc.collect()
            return None, None

        max_count = int(counts.max())
        max_extra = int(cfg.get("smote_max_per_class", 0))
        seed = _deep_seed_from_path(parquet_path, int(cfg.get("seed", 42)))
        rng = np.random.default_rng(seed)

        synth_list = []
        synth_labels = []

        for cls in range(3):
            cls_idx = np.flatnonzero(y_h == cls)
            if cls_idx.size == 0:
                continue

            need = max_count - int(cls_idx.size)
            if max_extra > 0:
                need = min(need, max_extra)
            if need <= 0:
                continue

            for _ in range(need):
                if cls_idx.size == 1:
                    i1 = i2 = int(cls_idx[0])
                else:
                    i1, i2 = rng.choice(cls_idx, size=2, replace=False)
                s1 = int(starts[int(i1)])
                s2 = int(starts[int(i2)])
                x1 = raw[s1 : s1 + seq_len]
                x2 = raw[s2 : s2 + seq_len]
                lam = float(rng.random())
                x_syn = x1 + lam * (x2 - x1)
                synth_list.append(x_syn.astype(np.float32, copy=False))
                synth_labels.append(y_day[int(i1)])

        if synth_list:
            synth_x = np.stack(synth_list, axis=0)
            synth_y = np.stack(synth_labels, axis=0).astype(np.int64, copy=False)

    class StableDayWindowDatasetWithSynthetic(Dataset):
        def __init__(
            self,
            raw: np.ndarray,
            starts: np.ndarray,
            base_labels: np.ndarray,
            seq_len: int,
            synth_x: np.ndarray | None,
            synth_y: np.ndarray | None,
        ):
            raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
            self.raw = raw
            self.starts = starts.astype(np.int64, copy=False)
            self.base_labels = base_labels.astype(np.int64, copy=False)
            self.synth_x = synth_x
            self.synth_y = synth_y
            self.seq_len = int(seq_len)

            feat_mean = raw.mean(axis=0, dtype=np.float64).astype(np.float32)
            feat_std = raw.std(axis=0, dtype=np.float64).astype(np.float32)
            feat_std = np.where(feat_std < 1e-6, 1.0, feat_std).astype(np.float32)
            self.feat_mean = feat_mean
            self.feat_std = feat_std

            if self.synth_x is None:
                self.labels = self.base_labels
            else:
                self.labels = np.concatenate([self.base_labels, self.synth_y], axis=0)

        def __len__(self) -> int:
            base_n = int(self.starts.size)
            synth_n = 0 if self.synth_x is None else int(self.synth_x.shape[0])
            return base_n + synth_n

        def __getitem__(self, idx: int):
            base_n = int(self.starts.size)
            if idx < base_n:
                s = int(self.starts[idx])
                x = self.raw[s : s + self.seq_len]
                y = self.base_labels[idx]
            else:
                j = idx - base_n
                x = self.synth_x[j]
                y = self.synth_y[j]

            x = (x - self.feat_mean) / self.feat_std
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
            x = np.clip(x, -10.0, 10.0)

            return torch.from_numpy(x.astype(np.float32, copy=False)), torch.from_numpy(y)

    ds = StableDayWindowDatasetWithSynthetic(
        raw=raw,
        starts=starts,
        base_labels=y_day,
        seq_len=seq_len,
        synth_x=synth_x,
        synth_y=synth_y,
    )
    stats = DayStats(
        ticker=os.path.basename(os.path.dirname(parquet_path)),
        file_name=os.path.basename(parquet_path),
        rows_raw=int(len(df)),
        rows_kept=int(len(ds)),
        step=int(step),
        max_rows=int(max_rows),
    )

    del df, y_float, labels, valid_mask, starts, y_day, class_mask
    gc.collect()
    return ds, stats


cfg_base = dict(DEEP_CONFIG)
base_suffix = str(cfg_base.get("result_suffix", "_stable_es"))

smote_train_config = {
    "result_suffix": "_smote_train_es",
    "balance_mode": "smote",
    "balance_horizon": int(cfg_base.get("horizons", [10])[0]),
    "balance_min_per_class": 2,
    "smote_max_per_class": 0,
}

smote_epochs = int(globals().get("DEEP_MAX_EPOCHS", 6))
smote_patience = int(globals().get("DEEP_EARLY_STOP_PATIENCE", 2))
smote_min_delta = float(globals().get("DEEP_EARLY_STOP_MIN_DELTA", 1e-4))

orig_builder = _deep_build_day_dataset_stable
orig_weights_fn = _deep_class_weights_stable
orig_loss_mode = DEEP_CONFIG.get("loss_mode")

try:
    _deep_build_day_dataset_stable = _deep_build_day_dataset_smote
    _deep_class_weights_stable = _deep_class_weights_uniform
    DEEP_CONFIG["loss_mode"] = "ce"

    smote_runs_out = run_all_deep_models_day_by_day_stable_earlystop(
        config=smote_train_config,
        max_epochs=smote_epochs,
        patience=smote_patience,
        min_delta=smote_min_delta,
    )
finally:
    _deep_build_day_dataset_stable = orig_builder
    _deep_class_weights_stable = orig_weights_fn
    if orig_loss_mode is None:
        DEEP_CONFIG.pop("loss_mode", None)
    else:
        DEEP_CONFIG["loss_mode"] = orig_loss_mode

base_runs_saved = _deep_load_saved_runs_from_results({**cfg_base, "result_suffix": base_suffix})
undersample_runs_saved = _deep_load_saved_runs_from_results({**cfg_base, "result_suffix": "_balanced_train_es"})
smote_runs = _deep_load_saved_runs_from_results({**cfg_base, "result_suffix": smote_train_config.get("result_suffix", "_smote_train_es")})

comparison_runs: dict = {}

if not base_runs_saved:
    print("No saved baseline runs found to compare against.")
elif not smote_runs:
    print("No saved SMOTE runs found. Run SMOTE training first.")
else:
    for arch, meta in smote_runs.items():
        base_meta = base_runs_saved.get(arch, {})
        under_meta = undersample_runs_saved.get(arch, {})

        comparison_runs[arch] = {
            "base_metrics": base_meta.get("test_metrics", {}),
            "undersample_metrics": under_meta.get("test_metrics", {}),
            "smote_metrics": meta.get("test_metrics", {}),
            "delta_smote_vs_base": _deep_compare_metrics(
                base_meta.get("test_metrics", {}),
                meta.get("test_metrics", {}),
                list(cfg_base.get("horizons", [])),
            ),
            "delta_smote_vs_undersample": _deep_compare_metrics(
                under_meta.get("test_metrics", {}),
                meta.get("test_metrics", {}),
                list(cfg_base.get("horizons", [])),
            ),
            "delta_undersample_vs_base": _deep_compare_metrics(
                base_meta.get("test_metrics", {}),
                under_meta.get("test_metrics", {}),
                list(cfg_base.get("horizons", [])),
            ),
        }

    timestamp = pd.Timestamp.now().isoformat()
    smote_suffix = str(smote_train_config.get("result_suffix", "_smote_train_es"))
    if smote_suffix and not smote_suffix.startswith("_"):
        smote_suffix = "_" + smote_suffix

    summary_path = os.path.join(cfg_base["results_dir"], f"deep_models_day_streaming_summary{smote_suffix}.json")

    smote_train_payload = {
        "timestamp": timestamp,
        "mode": "smote_train_oversample",
        "method": {
            "name": "smote_synthetic_oversample",
            "references": [
                "Chawla et al. 2002 (JAIR)",
                "He and Garcia 2009 (IEEE TKDE)",
            ],
        },
        "horizons": list(cfg_base.get("horizons", [])),
        "summary_path": summary_path,
        "runs": smote_runs,
    }

    comparison_payload = {
        "timestamp": timestamp,
        "mode": "comparison_smote_vs_undersample_vs_original",
        "horizons": list(cfg_base.get("horizons", [])),
        "runs": comparison_runs,
    }

    smote_train_path = os.path.join(cfg_base["results_dir"], f"deep_models_smote_train_results{smote_suffix}.json")
    compare_path = os.path.join(cfg_base["results_dir"], f"deep_models_smote_train_comparison_vs_original_undersample{smote_suffix}.json")

    with open(smote_train_path, "w") as f:
        json.dump(smote_train_payload, f, indent=2)
    with open(compare_path, "w") as f:
        json.dump(comparison_payload, f, indent=2)

    print(f"\nSMOTE training results saved -> {smote_train_path}")
    print(f"Comparison vs original/undersample saved -> {compare_path}")

# Advanced PyTorch Loss Functions
Defining `WeightedCrossEntropyLoss` and `FocalLoss` to better handle minority classes.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedCrossEntropyLoss(nn.Module):
    """
    Calculates class weights dynamically from the training labels (inverse frequency).
    """
    def __init__(self, weight=None):
        super().__init__()
        self.weight = weight

    def forward(self, inputs, targets):
        if self.weight is not None:
            self.weight = self.weight.to(inputs.device)
        return F.cross_entropy(inputs, targets, weight=self.weight)

class FocalLoss(nn.Module):
    """
    Implements Focal Loss to heavily penalize missing the minority classes 
    while ignoring the easy majority class.
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        if alpha is None:
            self.alpha = torch.tensor([1.0, 1.0, 1.0])
        else:
            self.alpha = torch.tensor(alpha, dtype=torch.float32)
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        
        self.alpha = self.alpha.to(inputs.device)
        alpha_t = self.alpha.gather(0, targets.view(-1)).view(targets.shape)
        
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [ ]:
# Let's utilize the WeightedCrossEntropyLoss and FocalLoss from above by updating the global 
# loss function logic used by the day-by-day continuous trainer.

def _deep_multihorizon_loss_advanced(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weight_tensors: list[torch.Tensor] | None = None,
    label_smoothing: float = 0.0,
    loss_mode: str | None = None,
    focal_gamma: float | None = None,
) -> torch.Tensor:
    if loss_mode is None:
        loss_mode = str(globals().get("DEEP_CONFIG", {}).get("loss_mode", "ce"))

    losses = []
    for i in range(logits.shape[1]):
        w = None if weight_tensors is None else weight_tensors[i]

        if loss_mode == "focal_advanced":
            # Using the FocalLoss class defined above
            if "advanced_focal_criterion" not in globals():
                globals()["advanced_focal_criterion"] = FocalLoss(gamma=2.0, reduction='mean')
            loss_i = globals()["advanced_focal_criterion"](logits[:, i, :], targets[:, i])
            
        elif loss_mode == "weighted_ce_advanced":
            # Using the WeightedCrossEntropyLoss defined above
            criterion = WeightedCrossEntropyLoss(weight=w)
            loss_i = criterion(logits[:, i, :], targets[:, i])
            
        elif loss_mode == "cb_focal":
            loss_i = _deep_focal_loss(
                logits[:, i, :],
                targets[:, i],
                weight=w,
                gamma=float(focal_gamma if focal_gamma is not None else 2.0),
                label_smoothing=float(label_smoothing),
            )
        else:
            loss_i = F.cross_entropy(
                logits[:, i, :],
                targets[:, i],
                weight=w,
                label_smoothing=float(label_smoothing),
            )
        losses.append(loss_i)
    return torch.stack(losses).mean()

# Monkey-patch the stable loss function to use our new advanced options
orig_deep_multihorizon_loss_stable = _deep_multihorizon_loss_stable
_deep_multihorizon_loss_stable = _deep_multihorizon_loss_advanced

# --- Run Training with Advanced Losses ---
cfg_base = dict(DEEP_CONFIG)

for adv_loss_mode, suffix in [("focal_advanced", "_focal_adv"), ("weighted_ce_advanced", "_wt_ce_adv")]:
    print(f"\n--- Training with advanced loss: {adv_loss_mode} ---")
    DEEP_CONFIG["loss_mode"] = adv_loss_mode
    
    adv_config = {
        "result_suffix": suffix,
        "loss_mode": adv_loss_mode
    }
    
    # We call the existing early-stop runner. 
    # It already implements robust epoch / file level check-pointing!
    run_all_deep_models_day_by_day_stable_earlystop(
        config=adv_config,
        max_epochs=int(globals().get("DEEP_MAX_EPOCHS", 6)),
        patience=int(globals().get("DEEP_EARLY_STOP_PATIENCE", 2)),
        min_delta=float(globals().get("DEEP_EARLY_STOP_MIN_DELTA", 1e-4)),
    )

# Restore original behavior
DEEP_CONFIG["loss_mode"] = orig_loss_mode
_deep_multihorizon_loss_stable = orig_deep_multihorizon_loss_stable


# Hybrid Data Sampling (ADASYN & SMOTETomek)
Applies ADASYN and SMOTETomek from imbalanced-learn to the dataset.
Because the data is 3D, it flattens it to 2D, applies sampling on a safe subsample (to avoid OOM), reshapes back to 3D, and caches to disk.

In [ ]:
import os
import gc
import numpy as np
from imblearn.over_sampling import ADASYN
from imblearn.combine import SMOTETomek

def apply_hybrid_sampling(X, y, method='adasyn', subsample_size=50000):
    n_samples, seq_len, features = X.shape
    
    # Safe subsample to prevent OOM
    if n_samples > subsample_size:
        idx = np.random.choice(n_samples, subsample_size, replace=False)
        X_sub = X[idx]
        y_sub = y[idx]
    else:
        X_sub = X
        y_sub = y

    # Flatten to 2D
    X_2d = X_sub.reshape(X_sub.shape[0], -1)
    
    if method == 'adasyn':
        sampler = ADASYN(random_state=42)
    elif method == 'smotetomek':
        sampler = SMOTETomek(random_state=42)
    else:
        raise ValueError(f"Unknown method {method}")
        
    print(f"Applying {method} on dataset of shape {X_2d.shape}...")
    X_res_2d, y_res = sampler.fit_resample(X_2d, y_sub)
    
    # Reshape back to 3D
    X_res_3d = X_res_2d.reshape(X_res_2d.shape[0], seq_len, features)
    
    print(f"Result shape: {X_res_3d.shape}")
    return X_res_3d, y_res

# Cache logic for ADASYN
dir_path = "sampled_data"
os.makedirs(dir_path, exist_ok=True)

X_adasyn_path = os.path.join(dir_path, 'X_train_adasyn.npy')
y_adasyn_path = os.path.join(dir_path, 'y_train_adasyn.npy')
X_smote_path = os.path.join(dir_path, 'X_train_smotetomek.npy')
y_smote_path = os.path.join(dir_path, 'y_train_smotetomek.npy')

# Example implementation if X_train_scaled and y_cls_train are loaded in memory:
# (Commented out initially so as not to crash if X_train_scaled is not in locals)

# if 'X_train_scaled' in locals() and 'y_cls_train' in locals():
#     if os.path.exists(X_adasyn_path) and os.path.exists(y_adasyn_path):
#         print("Loading cached ADASYN data...")
#         X_train_adasyn = np.load(X_adasyn_path)
#         y_train_adasyn = np.load(y_adasyn_path)
#     else:
#         X_train_adasyn, y_train_adasyn = apply_hybrid_sampling(X_train_scaled, y_cls_train[:, -1], 'adasyn')
#         np.save(X_adasyn_path, X_train_adasyn)
#         np.save(y_adasyn_path, y_train_adasyn)
#         
#     if os.path.exists(X_smote_path) and os.path.exists(y_smote_path):
#         print("Loading cached SMOTETomek data...")
#         X_train_smotetomek = np.load(X_smote_path)
#         y_train_smotetomek = np.load(y_smote_path)
#     else:
#         X_train_smotetomek, y_train_smotetomek = apply_hybrid_sampling(X_train_scaled, y_cls_train[:, -1], 'smotetomek')
#         np.save(X_smote_path, X_train_smotetomek)
#         np.save(y_smote_path, y_train_smotetomek)


# Native ML Models (XGBoost & Balanced Random Forest)
Train tree-based models that natively handle imbalance, predicting just the H=20 horizon.
Data is flattened to 2D for these models. Models are cached to disk using joblib.

In [ ]:
import joblib
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

xgb_model_path = os.path.join(dir_path, 'xgb_h20.pkl')
brf_model_path = os.path.join(dir_path, 'brf_h20.pkl')

# Example logic for when dataset is ready:
# if 'X_train_scaled' in locals() and 'y_cls_train' in locals() and 'X_test_scaled' in locals():
#     # We'll use the specific H=20 horizon. Suppose it is the last index in y_cls_train
#     # which we index as `target_idx = -1` (or whatever maps to H=20 in horizons array).
#     target_idx = -1 
#     
#     # Subset for faster training
#     n_samples_train = min(50000, len(X_train_scaled))
#     idx_train = np.random.choice(len(X_train_scaled), n_samples_train, replace=False)
#     
#     X_train_sub = X_train_scaled[idx_train]
#     y_train_sub = y_cls_train[idx_train, target_idx]
#     
#     X_test_sub = X_test_scaled
#     y_test_sub = y_cls_test[:, target_idx]
#     
#     # Flatten 3D to 2D
#     X_train_2d = X_train_sub.reshape(X_train_sub.shape[0], -1)
#     X_test_2d = X_test_sub.reshape(X_test_sub.shape[0], -1)
#     
#     # 1. XGBoost with class weights
#     if os.path.exists(xgb_model_path):
#         print("Loading existing XGBoost model...")
#         xgb_model = joblib.load(xgb_model_path)
#     else:
#         print("Training XGBoost...")
#         sample_weights = compute_sample_weight('balanced', y_train_sub)
#         xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
#         xgb_model.fit(X_train_2d, y_train_sub, sample_weight=sample_weights)
#         joblib.dump(xgb_model, xgb_model_path)
#         
#     # 2. Balanced Random Forest
#     if os.path.exists(brf_model_path):
#         print("Loading existing BalancedRandomForest model...")
#         brf_model = joblib.load(brf_model_path)
#     else:
#         print("Training BalancedRandomForest...")
#         brf_model = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
#         brf_model.fit(X_train_2d, y_train_sub)
#         joblib.dump(brf_model, brf_model_path)
#         
#     # Evaluate
#     print("--- XGBoost Evaluation ---")
#     y_pred_xgb = xgb_model.predict(X_test_2d)
#     print(classification_report(y_test_sub, y_pred_xgb))
#     
#     print("\n--- Balanced Random Forest Evaluation ---")
#     y_pred_brf = brf_model.predict(X_test_2d)
#     print(classification_report(y_test_sub, y_pred_brf))


# Autoencoder (Anomaly Detection Approach)
An LSTM Autoencoder trained to reconstruct ONLY Class 1 (Stationary) sequences. 
Rows in the test set with high Mean Squared Error (MSE) reconstruction loss can be classified as anomalies (Class 0 or Class 2) using an MSE threshold.

In [ ]:
import json

class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim=64):
        super(LSTMAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        
        # Encoder
        self.encoder = nn.LSTM(n_features, hidden_dim, batch_first=True)
        # Decoder
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        # Output layer
        self.output_layer = nn.Linear(hidden_dim, n_features)

    def forward(self, x):
        # Encode
        _, (hidden, _) = self.encoder(x)
        # Decoder input prep: repeat the hidden state to match the sequence length
        hidden_repeated = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        # Decode
        decoder_out, _ = self.decoder(hidden_repeated)
        # Reconstruct
        out = self.output_layer(decoder_out)
        return out


def train_lstm_autoencoder_day_by_day(
    cfg: dict,
    max_epochs: int = 5,
):
    """
    Robust day-by-day training loop for the LSTM Autoencoder with per-file, per-epoch checkpointing.
    Train ONLY on Class 1 (Stationary) sequences.
    """
    # Setup
    os.makedirs(cfg.get('weights_dir', 'model_weights'), exist_ok=True)
    os.makedirs(cfg.get('results_dir', 'results'), exist_ok=True)
    
    arch = "lstm_autoencoder"
    checkpoint_path = os.path.join(cfg.get('weights_dir', 'model_weights'), f"{arch}_checkpoint.pt")
    weights_path = os.path.join(cfg.get('weights_dir', 'model_weights'), f"{arch}_weights.pt")
    results_path = os.path.join(cfg.get('results_dir', 'results'), f"{arch}_results.json")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Needs to match features
    input_dim = len(DEEP_RAW_LOB_10_COLS)
    seq_len = int(cfg.get('seq_len', 10))
    
    model = LSTMAutoencoder(seq_len=seq_len, n_features=input_dim, hidden_dim=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    tickers = _deep_resolve_tickers(cfg)
    files_by_ticker = _deep_collect_files_by_ticker(cfg.get('data_dir', 'data/processed'), tickers, cfg.get('max_files_per_ticker', 0))
    train_files, eval_files = _deep_split_train_eval_files(files_by_ticker, float(cfg.get('train_file_fraction', 0.8)))
    
    start_epoch = 1
    start_file_idx = 1
    best_loss = float('inf')
    total_train_days = 0
    epoch_history = []
    
    if os.path.exists(checkpoint_path):
        print(f"[{arch}] Found checkpoint at {checkpoint_path}, resuming...")
        chk = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(chk['model'])
        optimizer.load_state_dict(chk['optimizer'])
        start_epoch = chk['epoch']
        start_file_idx = chk['file_idx'] + 1
        best_loss = chk.get('best_loss', float('inf'))
        epoch_history = chk.get('epoch_history', [])
        total_train_days = chk.get('total_train_days', 0)
        
        if start_file_idx > len(train_files):
            start_epoch += 1
            start_file_idx = 1
    
    def save_checkpoint(ep, f_idx):
        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'epoch': ep,
            'file_idx': f_idx,
            'best_loss': best_loss,
            'epoch_history': epoch_history,
            'total_train_days': total_train_days,
        }, checkpoint_path)

    target_h_idx = 0 # Using first horizon as the target

    for epoch in range(start_epoch, max_epochs + 1):
        model.train()
        print(f"\n[{arch}] Epoch {epoch}/{max_epochs}")
        
        loss_sum = 0.0
        good_batches = 0
        
        for file_idx, (ticker, path) in enumerate(train_files, start=1):
            if file_idx < start_file_idx:
                continue
                
            ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=True)
            if ds is None:
                continue
                
            # Filter ONLY class 1 (Stationary)
            mask_c1 = (ds.labels[:, target_h_idx] == 1)
            raw_c1 = ds.raw[ds.starts[mask_c1]] # Wait, ds.raw is the full day matrix, not windows!
            starts_c1 = ds.starts[mask_c1]
            labels_c1 = ds.labels[mask_c1]
            
            if starts_c1.size == 0:
                continue
                
            ds.starts = starts_c1
            ds.labels = labels_c1
            loader = _deep_make_loader(ds, cfg, is_train=True)
            
            for xb, _ in loader:
                xb = xb.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                
                outputs = model(xb)
                loss = criterion(outputs, xb)
                
                if not torch.isfinite(loss):
                    continue
                    
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                loss_sum += loss.item()
                good_batches += 1
            
            total_train_days += 1
            save_checkpoint(epoch, file_idx)
            
            del ds, loader
            _deep_cleanup_cuda()
            
        start_file_idx = 1
        avg_loss = loss_sum / max(1, good_batches)
        print(f"[{arch}] Epoch {epoch} complete, Avg Loss: {avg_loss:.6f}")
        
        epoch_history.append({'epoch': epoch, 'avg_train_loss': avg_loss})
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            # Save best weights
            torch.save(model.state_dict(), weights_path)
            print(f"[{arch}] Saved new best weights to {weights_path}")
            
        # Optional eval step here
        
    print(f"[{arch}] Training complete. Best Loss: {best_loss:.6f}")
    
    # Save results
    res = {
        'architecture': arch,
        'epoch_history': epoch_history,
        'best_train_loss': best_loss,
        'total_train_days': total_train_days,
        'checkpoint_used': checkpoint_path,
        'weights_path': weights_path
    }
    with open(results_path, "w") as f:
        json.dump(res, f, indent=2)

if "DEEP_CONFIG" in globals():
    # Only run training if we are executing globally (i.e. notebook execution)
    train_lstm_autoencoder_day_by_day(DEEP_CONFIG, max_epochs=2)
